In [ ]:
# Dataset: SynTTS-Commands (English, Free-ST subset)
# https://huggingface.co/datasets/lugan/Syntts-Commands-Media-Dataset
# Hanya 4 kelas playback control yang dipakai.

!pip install -q huggingface_hub

In [ ]:
import os
import zipfile
from huggingface_hub import hf_hub_download

REPO_ID = "lugan/Syntts-Commands-Media-Dataset"
SUBSET = "Free_ST_English"   # ~190 MB untuk 4 kelas. "VoxCeleb12_English" jauh lebih besar (~1.7 GB)

classes = [
    "Play",
    "Pause",
    "Next_track",
    "Previous_track"
]

DATASET_PATH = "/content/dataset/dataset"
os.makedirs(DATASET_PATH, exist_ok=True)

for label in classes:

    if os.path.isdir(os.path.join(DATASET_PATH, label)):
        print(f"{label}: sudah ada, skip")
        continue

    zip_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=f"{SUBSET}/{label}.zip"
    )

    # Setiap zip berisi folder <label>/*.wav
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATASET_PATH)

    print(f"{label}: {len(os.listdir(os.path.join(DATASET_PATH, label)))} file")

print("Dataset berhasil diunduh dan diekstrak!")

In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    level = root.replace("/content/dataset", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    if level >= 1:
        for file in files[:5]:
            print(f"{indent}    {file}")

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

records = []

for label in classes:

    folder = os.path.join(DATASET_PATH, label)

    for filename in os.listdir(folder):

        if filename.lower().endswith((".wav", ".mp3", ".flac", ".ogg")):

            filepath = os.path.join(folder, filename)

            try:
                y, sr = librosa.load(filepath, sr=None, mono=False)

                if y.ndim == 1:
                    channels = 1
                    duration = len(y) / sr
                else:
                    channels = y.shape[0]
                    duration = y.shape[1] / sr

                records.append({
                    "file": filepath,
                    "label": label,
                    "sample_rate": sr,
                    "channels": channels,
                    "duration": duration
                })

            except Exception as e:
                print("ERROR:", filepath)
                print(e)

df = pd.DataFrame(records)

print(df.head())
print()
print("Jumlah data:", len(df))

In [ ]:
print(df["label"].value_counts())

In [ ]:
print(df["sample_rate"].value_counts())

In [ ]:
print(df["channels"].value_counts())

In [ ]:
print(df["duration"].describe())

In [ ]:
import os

for label in classes:
    folder = os.path.join(DATASET_PATH, label)

    print(f"\n===== {label} =====")

    files = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".wav", ".mp3", ".flac", ".ogg"))
    ]

    for f in files[:20]:
        print(f)

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "metadata" in file.lower():
            print(os.path.join(root, file))

In [ ]:
import re

def extract_speaker_id(filename):
    match = re.search(r'P(\d+)', filename)
    if match:
        return match.group(1)
    return None

In [ ]:
df["speaker_id"] = df["file"].apply(
    lambda x: extract_speaker_id(os.path.basename(x))
)

print(df[["file", "speaker_id", "label"]].head(20))

In [ ]:
print("Jumlah speaker:")
print(df["speaker_id"].nunique())

In [ ]:
speaker_command = df.groupby("speaker_id")["label"].nunique()

print(speaker_command.value_counts())

In [ ]:
print(
    df.groupby("speaker_id")["label"]
      .nunique()
      .sort_values(ascending=False)
      .head(20)
)

In [ ]:
example_speaker = df["speaker_id"].value_counts().index[0]

print(
    df[df["speaker_id"] == example_speaker]
    [["file", "speaker_id", "label"]]
    .to_string(index=False)
)

#Speaker-Independent Split

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

print("Jumlah data:", len(df))
print("\nDistribusi label:")
print(df["label"].value_counts())

# ============================================================
# 1. SPLIT BERDASARKAN SPEAKER
#    70% Train, 30% Temp
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        df,
        y=df["label"],
        groups=df["speaker_id"]
    )
)

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

# ============================================================
# 2. Temp → Validation + Test
#    15% Validation, 15% Test
# ============================================================

gss_val_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss_val_test.split(
        temp_df,
        y=temp_df["label"],
        groups=temp_df["speaker_id"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# ============================================================
# 3. SHUFFLE
# ============================================================

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# ============================================================
# 4. HASIL SPLIT
# ============================================================

print("\n========== HASIL SPLIT ==========")

print(f"Train      : {len(train_df)} audio")
print(f"Validation : {len(val_df)} audio")
print(f"Test       : {len(test_df)} audio")

print("\n========== DISTRIBUSI LABEL ==========")

print("\nTRAIN:")
print(train_df["label"].value_counts())

print("\nVALIDATION:")
print(val_df["label"].value_counts())

print("\nTEST:")
print(test_df["label"].value_counts())

# ============================================================
# 5. CEK SPEAKER
# ============================================================

train_speakers = set(train_df["speaker_id"])
val_speakers = set(val_df["speaker_id"])
test_speakers = set(test_df["speaker_id"])

print("\n========== SPEAKER ==========")

print("Train speaker      :", len(train_speakers))
print("Validation speaker :", len(val_speakers))
print("Test speaker       :", len(test_speakers))

print("\n========== SPEAKER OVERLAP ==========")

print("Train ∩ Validation :", len(train_speakers & val_speakers))
print("Train ∩ Test       :", len(train_speakers & test_speakers))
print("Validation ∩ Test  :", len(val_speakers & test_speakers))

In [ ]:
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

print("Split berhasil disimpan.")

#audio preprocessing

In [ ]:
!pip install librosa soundfile

In [ ]:
TARGET_SR = 16000
DURATION = 1.0
TARGET_LENGTH = int(TARGET_SR * DURATION)

N_MFCC = 40
N_FFT = 512
HOP_LENGTH = 160
WIN_LENGTH = 400

print("Target sample rate :", TARGET_SR)
print("Target samples     :", TARGET_LENGTH)
print("MFCC               :", N_MFCC)
print("n_fft              :", N_FFT)
print("hop_length         :", HOP_LENGTH)
print("win_length         :", WIN_LENGTH)

In [ ]:
import librosa
import numpy as np

def preprocess_audio(file_path, label=None, random_crop=False):
    """
    Load dan preprocess audio:
    - Mono
    - Resample ke 16 kHz
    - Normalize amplitude
    - Audio menjadi 1 detik

    Untuk Unknown:
    - Jika audio > 1 detik, dapat mengambil random crop 1 detik.
    """

    # Load audio
    audio, sr = librosa.load(
        file_path,
        sr=TARGET_SR,
        mono=True
    )

    # Normalize amplitude
    max_val = np.max(np.abs(audio))

    if max_val > 0:
        audio = audio / max_val

    # ========================================================
    # Audio lebih pendek dari 1 detik
    # ========================================================

    if len(audio) < TARGET_LENGTH:

        pad_length = TARGET_LENGTH - len(audio)

        audio = np.pad(
            audio,
            (0, pad_length),
            mode="constant"
        )

    # ========================================================
    # Audio lebih panjang dari 1 detik
    # ========================================================

    elif len(audio) > TARGET_LENGTH:

        if label == "Unknown" and random_crop:

            # Random crop 1 detik
            max_start = len(audio) - TARGET_LENGTH

            start = np.random.randint(
                0,
                max_start + 1
            )

            audio = audio[
                start:start + TARGET_LENGTH
            ]

        else:

            # Untuk command:
            # tetap mengambil bagian awal
            audio = audio[:TARGET_LENGTH]

    return audio

In [ ]:
# Command
command_sample = train_df[
    train_df["label"] == "Play"
].iloc[0]

audio_command = preprocess_audio(
    command_sample["file"],
    label=command_sample["label"]
)

print("COMMAND")
print("File        :", command_sample["file"])
print("Shape       :", audio_command.shape)
print("Sample rate :", TARGET_SR)
print("Duration    :", len(audio_command) / TARGET_SR)
print("Min         :", audio_command.min())
print("Max         :", audio_command.max())

In [ ]:
import matplotlib.pyplot as plt
import librosa.display

plt.figure(figsize=(12, 4))

librosa.display.waveshow(
    audio_command,
    sr=TARGET_SR
)

plt.title("Waveform Command Setelah Preprocessing")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.show()

#MFCC

In [ ]:
def extract_mfcc(file_path, label=None, random_crop=False):
    """
    Preprocessing audio + ekstraksi MFCC.

    Output:
        shape = (N_MFCC, time_frames)
    """

    audio = preprocess_audio(
        file_path,
        label=label,
        random_crop=random_crop
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=TARGET_SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH
    )

    return mfcc

In [ ]:
# Ambil satu file COMMAND
command_row = train_df[train_df["label"] == "Play"].iloc[0]


# =========================
# COMMAND
# =========================
mfcc_command = extract_mfcc(
    command_row["file"],
    label=command_row["label"],
    random_crop=False
)

print("COMMAND")
print("File:", command_row["file"])
print("MFCC shape:", mfcc_command.shape)
print("MFCC min:", mfcc_command.min())
print("MFCC max:", mfcc_command.max())

In [ ]:
plt.figure(figsize=(10, 5))

librosa.display.specshow(
    mfcc_command,
    sr=TARGET_SR,
    hop_length=HOP_LENGTH,
    x_axis="time"
)

plt.colorbar(format="%+2.0f dB")
plt.title("MFCC - Play")
plt.xlabel("Time")
plt.ylabel("MFCC Coefficient")

plt.show()

In [ ]:
for label in classes:

    sample = train_df[
        train_df["label"] == label
    ].iloc[0]["file"]

    mfcc = extract_mfcc(sample)

    print(f"{label:15s} -> {mfcc.shape}")

In [ ]:
print("NaN :", np.isnan(mfcc).any())
print("Inf :", np.isinf(mfcc).any())

#PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Mapping label
class_names = classes

label_to_idx = {
    label: idx
    for idx, label in enumerate(class_names)
}

print(label_to_idx)

In [ ]:
class SpeechCommandDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Ekstraksi MFCC
        mfcc = extract_mfcc(row["file"])

        # Convert numpy -> torch tensor
        mfcc = torch.tensor(
            mfcc,
            dtype=torch.float32
        )

        # Tambahkan channel dimension
        # [40, 101] -> [1, 40, 101]
        mfcc = mfcc.unsqueeze(0)

        # Label
        label = label_to_idx[row["label"]]

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return mfcc, label

In [ ]:
train_dataset = SpeechCommandDataset(train_df)
val_dataset = SpeechCommandDataset(val_df)
test_dataset = SpeechCommandDataset(test_df)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
X, y = train_dataset[0]

print("Input shape :", X.shape)
print("Label       :", y)
print("Label name  :", class_names[y.item()])

In [ ]:
X_batch, y_batch = next(iter(train_loader))

print("Batch input :", X_batch.shape)
print("Batch label :", y_batch.shape)

#DS-CNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Depthwise Separable Convolution Block

In [ ]:
class DepthwiseSeparableConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):
        super().__init__()

        # Depthwise convolution
        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(in_channels)

        # Pointwise convolution
        self.pointwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.depthwise(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu(x)

        return x

In [ ]:
class DSCNN(nn.Module):

    def __init__(self, num_classes=4):
        super().__init__()

        # Initial convolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        # DS-CNN blocks
        self.block1 = DepthwiseSeparableConv(
            32,
            64,
            stride=2
        )

        self.block2 = DepthwiseSeparableConv(
            64,
            128,
            stride=2
        )

        self.block3 = DepthwiseSeparableConv(
            128,
            128,
            stride=2
        )

        self.block4 = DepthwiseSeparableConv(
            128,
            256,
            stride=2
        )

        # Global Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout
        self.dropout = nn.Dropout(0.3)

        # Classifier
        self.fc = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):

        x = self.conv1(x)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.global_pool(x)

        # [batch, 256, 1, 1]
        # ->
        # [batch, 256]
        x = torch.flatten(x, 1)

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [ ]:
model = DSCNN(
    num_classes=len(class_names)
).to(device)

print(model)

In [ ]:
X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)

with torch.no_grad():
    output = model(X_batch)

print("Input shape :", X_batch.shape)
print("Output shape:", output.shape)

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Class weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_labels = train_df["label"].map(
    label_to_idx
).values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels
)

print("Class weights:")

for name, weight in zip(
    class_names,
    class_weights
):
    print(f"{name:15s}: {weight:.4f}")

In [ ]:
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(class_weights)

Loss Funtion

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

Optimizer

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device)
        y = y.to(device)

        # Reset gradient
        optimizer.zero_grad()

        # Forward
        outputs = model(X)

        # Loss
        loss = criterion(outputs, y)

        # Backward
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * X.size(0)

        _, predicted = torch.max(outputs, 1)

        total += y.size(0)
        correct += (predicted == y).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, y in loader:

            X = X.to(device)
            y = y.to(device)

            # Forward
            outputs = model(X)

            # Loss
            loss = criterion(outputs, y)

            # Statistics
            running_loss += loss.item() * X.size(0)

            _, predicted = torch.max(outputs, 1)

            total += y.size(0)
            correct += (predicted == y).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
NUM_EPOCHS = 30

PATIENCE = 5

best_val_loss = float("inf")
patience_counter = 0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

In [ ]:
import time

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    # =========================
    # TRAIN
    # =========================

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # =========================
    # VALIDATION
    # =========================

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    # =========================
    # SAVE HISTORY
    # =========================

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start_time

    # =========================
    # PRINT
    # =========================

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.4f} "
        f"| Time: {elapsed:.1f}s"
    )

    # =========================
    # BEST CHECKPOINT
    # =========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            "/content/best_dscnn.pth"
        )

        print("  ✓ Best model saved!")

    else:

        patience_counter += 1

        print(
            f"  No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

    # =========================
    # EARLY STOPPING
    # =========================

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")

        break

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/best_dscnn.pth",
        map_location=device
    )
)

model.to(device)

print("Best model loaded.")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history["train_loss"],
    label="Train Loss"
)

plt.plot(
    epochs,
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history["train_acc"],
    label="Train Accuracy"
)

plt.plot(
    epochs,
    history["val_acc"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid()

plt.show()

In [ ]:
import numpy as np

def get_predictions(model, loader, device):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for X, y in loader:

            X = X.to(device)

            outputs = model(X)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                y.numpy()
            )

    return (
        np.array(all_labels),
        np.array(all_predictions)
    )

In [ ]:
y_true, y_pred = get_predictions(
    model,
    test_loader,
    device
)

print("Jumlah test samples:", len(y_true))
print("Jumlah predictions :", len(y_pred))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("========== TEST RESULTS ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

In [ ]:
print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - DS-CNN")

plt.tight_layout()
plt.show()